# XAI — sparse-transcoder attribution on the KB+QA arm

**Purpose.** Reproduce, on the Kineret cohort, the token-level
attribution analysis reported for INTERVenE-Enc on MIMIC-IV (AAAI
companion paper). Specifically, aggregate per-token gradient-decoupled
sparse-transcoder contributions from the deployed KB+QA arm across
~300 test patients, per outcome, and identify which named clinical
concepts and QA `*_PATTERN` tokens the model is reading.

**Status.** This notebook is a scaffold. The XAI pipeline
(transcoder training + gradient-decoupled hooking + attribution
aggregation) is not yet vendored into `kineret/intervene/`. Two
sources for the code:

- **The AAAI code snapshot**, at
  `../../papers-drafts/AAAI2027/AAAI_2027___INTERVenE/code/INTERVenE/encoder/xai/`,
  contains `transcoder/`, `attribute.py`, `feature_concepts.py`, and a
  demo notebook (`xai_demo.ipynb`). That directory is the authoritative
  source and matches the MIMIC-IV numbers in the AAAI paper.
- **The previous ss-STraTS XAI run**, at
  `results/transformers-results-old-with-xai/QA/xai.ipynb`, is retained
  for reference. It runs the same attribution paradigm on the pre-ladder
  ss-STraTS variants, not on the seven-arm KB+QA arm; it is superseded
  by this notebook.

The steps below name what needs to happen; the vendoring step (copying
the AAAI `xai/` module into `kineret/intervene/xai/` and threading it
into a runnable pipeline) is the AIIM tasks-list item that gates this
notebook.

## Pipeline

1. **Load the trained KB+QA checkpoint** from
   `checkpoints/intervene_kb_qa/ckpt_best.pt` (produced by
   `benchmark.run_ladder`).
2. **Rebuild the test dataloader** for the KB+QA arm, keyed on the same
   test patients used in the canonical run.
3. **Train sparse transcoders** on the encoder's per-block MLP inputs.
   Layer-by-layer, using a JumpReLU objective under a
   reconstruction + L0 loss (AAAI paper §Interpretability). Target
   $R^2 \gtrsim 0.97$ per layer.
4. **Attach the gradient-decoupled hooks** described by the identity
   $y^{\mathrm{used}}_L = y^{\mathrm{true}}_L + (\hat{y}_L - \mathrm{sg}(\hat{y}_L))$
   so the forward pass is bit-exact to the deployed model.
5. **Compute per-token contributions** — input × gradient of the risk
   logit through the transcoder feature basis, summed across encoder
   layers. Aggregate across ~300 test patients per outcome.
6. **Plot** the top-$k$ positive / negative contributors per outcome
   (analogue of Figure `xai_death.png` in the AAAI paper).
7. **Export** the aggregated attribution tables and figure to
   `outputs/figures/xai/` for the AIIM paper.

The QA-specific interest is whether `*_PATTERN` tokens appear among
the top contributors per outcome, matching the qualitative claim in
the AIIM discussion (`results.tex` §QA).

In [ ]:
# ---------------------------------------------------------------
# Scaffold. Un-comment cell-by-cell once the xai module lands in
# kineret/intervene/xai/ (see notebook intro).
# ---------------------------------------------------------------
#
# from kineret.intervene.xai.transcoder import fit_transcoders
# from kineret.intervene.xai.attribute  import gradient_decoupled_contrib
# from kineret.intervene.xai.feature_concepts import group_by_concept
#
# 1. Load checkpoint + dataloader.
# ckpt = paths.CHECKPOINT_DIR / "intervene_kb_qa" / "ckpt_best.pt"
# model, tokenizer, _ = load_intervene_checkpoint(ckpt)
# test_loader = build_test_loader(kind='kb+qa')
#
# 2. Fit per-layer transcoders.
# tcs = fit_transcoders(model, test_loader, layers=range(model.n_layers))
#
# 3. Attach gradient-decoupled hooks.
# hooked_model = attach_gd_hooks(model, tcs)
#
# 4. Aggregate contributions across N patients.
# contribs = gradient_decoupled_contrib(hooked_model, test_loader,
#                                        outcomes=OUTCOMES, n_patients=300)
#
# 5. Group and plot.
# grouped = group_by_concept(contribs, vocabulary=tokenizer.vocab)
# plot_top_drivers(grouped)
print("Scaffold only -- see notebook intro for the vendoring step.")